In [ ]:
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as mpe
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import rasterio as rio
import seaborn as sns
from tqdm import tqdm
import xarray as xr
from scipy.stats import mannwhitneyu, ttest_ind, wilcoxon

from project.utils.data import enrich, load_hillshade, sel_subset
from project.utils.metrics import compute_metrics
from project.utils.plotting import ProjectCustomExtent, despine

plt.rcParams["figure.dpi"] = 150

# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

ToDo

- [x] Use discharge instead of streamflow
- [x] Not trend analysis
- [x] Perforemance: only biases test + correlation
- [x] Splitting maps: not color for RS catchments
- [x] Overfitting test: move to appendix, show point-wise difference, report paired test
- [x] Retention: Map and one-two scatterplots, retention on y-axis
- [x] PREVAH vs runoff module. Appendix?
- [x] Show streamflow maps for 2021-07-15: discharge and percentiles, runoff, temporal evolution
- [ ] Time series: show prevah, runoff, discharge

Optional
- [ ] Test pretraining with lumped
- [ ] Run training for many epochs


In [ ]:
# Load predictions.
ds = xr.open_zarr('/net/argon/landclim/kraftb/machflow_routing/data/results/predictions.zarr').sel(
    time=slice('1993', None))

# Load HRU polygons.
hrus = gpd.read_file(
    '/net/argon/landclim/kraftb/machflow_routing/data/processed/hrus/hrus.gpkg', use_arrow=True).set_index('EZGNR')

metrics_max = ['kge', 'r']
metrics_min = ['kge_alpha', 'kge_beta', 'flv', 'fhv']

# metric_nice_names = {
#     'bias2': 'Bias²',
#     'varerr': 'Variance error',
#     'phaseerr': 'Phase error',
#     'nse': 'Nash-Sutcliffe Efficiency',
#     'nnse': 'Normalized Nash-Sutcliffe Efficiency',
#     'kge': 'Kling-Gupta Efficiency',
#     'r': 'Pearson\'s r',
#     'relbias': '%Bias',
#     'flv': '%Low Flow Bias (<Q5)',
#     'fhv': '%High Flow Bias (>Q98)',
# }

# Compute metrics and add them to dataset.
met = compute_metrics(ds.Q_obs_m3s, ds.Q_pred_m3s, metrics=metrics_min + metrics_max, dim='time')
ds = xr.merge([ds, met])

for m in metrics_min + metrics_max:
    name = f'{m}_impr'

    # Create the target variable once per metric.
    # Use the metric itself as a template (and make sure it's float so NaN works).
    if name not in ds:
        ds[name] = xr.full_like(ds[m], np.nan, dtype=float).load()

    for model in ds.model.values:
        # Pair model with the corresponding other model
        model_type, split_type = model.split('_')
        other_model_type = 'routed' if model_type == 'lumped' else 'lumped'
        other_model = f'{other_model_type}_{split_type}'
        if other_model not in ds.model.values:
            continue

        value_model = ds.sel(model=model)[m]
        value_other = ds.sel(model=other_model)[m]

        if m in metrics_min:
            value_model = np.abs(value_model)
            value_other = np.abs(value_other)

        if m in metrics_min:
            # Lower is better
            pec_impr = (value_other - value_model) / (value_other + 1e-6) * 100.0
        else:
            # Higher is better
            pec_impr = (value_model - value_other) / (1 - value_other) * 100.0

        # Write only the slice for this model
        ds[name].loc[dict(model=model)] = pec_impr

rivers = gpd.read_file('/net/argon/landclim/kraftb/machflow_routing/data/raw/rivernet', use_arrow=True)

rivers_subset = rivers.loc[rivers.CUM_LEN > 20000]

bg = load_hillshade(
    hillshade_dir='/net/argon/landclim/kraftb/machflow_routing/data/processed/',
    hrus=hrus,
    ds=ds,
    use_terrain=False,
    force_recompute=False)

# Performance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 2), sharey=True, sharex=True, gridspec_kw={'hspace': 0.25})

ds_m = sel_subset(ds, 3)['kge']

for j, (split_type, split_name) in enumerate(zip(['random', 'easy', 'hard'], ['RS', 'CS-E', 'CS-H'])):
    ax = axes[j]

    lumped_met = 0.0
    perc_impr = 0.0

    for t, model_type in enumerate(['lumped', 'routed']):

        model = f'{model_type}_{split_type}'

        ds_m_subset = ds_m.sel(model=model)
        ds_m_subset = ds_m_subset[np.isfinite(ds_m_subset)]

        colors = {
            'lumped': "#4F9CFF",
            'routed': "#DAA507",
        }

        # styles = {
        #     'random': dict(linestyle='-'),
        #     'easy': dict(linestyle='--'),
        #     'hard': dict(linestyle=':'),
        # }

        color = colors[model_type]

        model_type_new = 'LSTM (lumped)' if model_type != 'routed' else 'DROP (routed)'

        ax.ecdf(
            ds_m_subset,
            lw=1.2,
            color=color,
            label=f'{model_type_new}'
        )

        #ax.axvline(0, color='k', ls='--', dashes=(5, 5), lw=0.4, zorder=999)

        median = ds_m_subset.median()
        if model_type == 'lumped':
            lumped_met = median.item()
            disp_txt = f'Median={lumped_met:0.2f}'
        else:
            perc_impr = (median.item() - lumped_met) / lumped_met * 100.0
            if perc_impr > 0:
                sign = '+'
            else:
                sign = '-'
            disp_txt = f'Median={median.item():0.2f}\n({sign}{perc_impr:0.1f}%)'

        ax.axvline(median, ls='--', lw=1.2, color=color)
        ax.text(0.02, [0.95, 0.85][t], disp_txt, color=color,
                va='top', ha='left', transform=ax.transAxes, fontsize=8, weight='bold')

    if j == 0:
        ax.set_ylabel('Cumulative Density', y=0.5)
    else:
        ax.set_ylabel('')

    if j == 1:
        ax.set_xlabel(ds_m.attrs['long_name'])

    ax.set_xlim(-1, 1)

    ax.set_title(f'Split type: {split_name}', fontsize=10)

    despine(ax)

axes[0].legend(
    frameon=False, handlelength=1, title='Model type', loc='center left', bbox_to_anchor=(0, 0.5),
    fontsize=8, title_fontsize=9)

fig.savefig('./figures/kge.pdf', dpi=300, bbox_inches='tight')

In [ ]:
metrics = ['r', 'kge_alpha', 'kge_beta', 'flv', 'fhv']

fig, axes = plt.subplots(1, len(metrics), figsize=(10, 2),
                         sharex=True, sharey=False,
                         gridspec_kw={'wspace': 0.4})


df = sel_subset(
    ds.where(ds.has_obs.load(), drop=True), 3)[metrics].to_dataframe(
        ).reset_index()
parts = df['model'].str.rsplit('_', n=1, expand=True)
df['Model type'] = parts[0].str.capitalize()
df['Split type'] = parts[1].str.capitalize()
df['Model type'] = df['Model type'].replace('Lumped', 'LSTM (lumped)')
df['Model type'] = df['Model type'].replace('Routed', 'DROP (routed)')

for m, metric in enumerate(metrics):

    ax = axes[m]

    colors = {
        'LSTM (lumped)': "#4F9CFF",
        'DROP (routed)': "#DAA507",
    }

    # common kwargs for both panels
    box_kws = dict(
        data=df, x='Split type', y=metric, hue='Model type',
        palette=colors, showfliers=False, gap=0.1
    )

    # draw the same plot twice (bottom zoom + top zoom)
    # legend only on the top row, last metric (bottom axis) to place later
    want_legend = m == 0
    sns.boxplot(ax=ax, order=['Random', 'Easy', 'Hard'], legend=want_legend, **box_kws)

    ax.set_xticklabels(['RS', 'CS-E', 'CS-H'])

    metric_nice = ds[metric].attrs['long_name']

    # titles / labels
    ax.set_title(metric_nice, fontsize=8)

    ax.set_ylabel('')

    # reference line (draw on both; it will only show if within the panel range)
    # for a in (ax_bot, ax_top):
    #     a.axhline(1, color='k', ls='--', lw=0.8, zorder=0.001)

    # optional: your despine helper on both axes
    despine(ax)

    ax.axhline(1, color='k', ls='--', lw=0.8, zorder=999)

    if want_legend:
        legend_ax = ax

    if m == 2:
        ax.set_xlabel('Split type')
    else:
        ax.set_xlabel('')

# # place the legend outside using the saved axis
if legend_ax is not None:
    sns.move_legend(legend_ax, loc='upper left', ncol=2, title='',
                    bbox_to_anchor=(-0.08, -0.13), frameon=False, fontsize=8, title_fontsize=9)

fig.savefig('./figures/biases.pdf', dpi=300, bbox_inches='tight')

In [ ]:
metrics = ['bias2', 'varerr', 'phaseerr']

# --- helper: replace an Axes with two stacked Axes that share x ---
def make_broken_axes(fig, ax, hspace=0.06, height_ratios=(1, 1)):
    pos = ax.get_position()
    fig.delaxes(ax)  # remove original
    x0, y0, w, h = pos.x0, pos.y0, pos.width, pos.height
    total = sum(height_ratios)
    gap = hspace * h

    h_top = h * (height_ratios[0] / total) - gap / 2
    h_bot = h * (height_ratios[1] / total) - gap / 2

    ax_top = fig.add_axes([x0, y0 + h - h_top, w, h_top])
    ax_bot = fig.add_axes([x0, y0,              w, h_bot])
    return ax_top, ax_bot

# --- helper: draw break marks on the touching spines ---
def add_break_marks(ax_top, ax_bot, size=0.5, lw=1.0, color="0.3"):
    # draw only on the LEFT side
    kwargs = dict(markersize=10, color=color, markeredgewidth=lw,
                  linestyle="none", clip_on=False)
    marker = [(-1, -size), (1, size)]  # small diagonal

    # top axis: bottom-left corner (x=0,y=0 in axes coords)
    ax_top.plot([0], [0], transform=ax_top.transAxes, marker=marker, **kwargs)
    # bottom axis: top-left corner (x=0,y=1)
    ax_bot.plot([0], [1], transform=ax_bot.transAxes, marker=marker, **kwargs)


# --- choose per-metric ranges (tune these as you like) ---
# lower windows focus on the bulk; upper windows show the extreme tail
breaks = {
    m: (
        (0,  190),                                      # lower y-limits
        (418, 440)       # upper y-limits
    )
    for m in metrics
}

fig, axes = plt.subplots(2, len(metrics), figsize=(10, 6),
                         sharex=False, sharey=False,
                         gridspec_kw={'wspace': 0.4})

legend_ax = None  # we’ll save the bottom axis of the top-right panel for legend

for i, subset in enumerate(['Train', 'Test']):
    for m, metric in enumerate(metrics):

        df = sel_subset(
            ds.where(ds.has_obs.load(), drop=True), 1 if subset == 'Train' else 3)[metrics].to_dataframe(
                ).reset_index()
        parts = df['model'].str.rsplit('_', n=1, expand=True)
        df['Model type'] = parts[0].str.capitalize()
        df['Split type'] = parts[1].str.capitalize()

        # Get breaks.
        lower_ylim, upper_ylim = breaks[metric]

        # replace this slot with broken axes
        bot_h = lower_ylim[1] - lower_ylim[0]
        top_h = upper_ylim[1] - upper_ylim[0]
        ratio = top_h / bot_h
        ax = axes[i, m]
        ax_top, ax_bot = make_broken_axes(fig, ax, hspace=0.04, height_ratios=(ratio, 1))

        colors = {
            'Lumped': "#4F9CFF",
            'Routed': "#DAA507",
        }

        # common kwargs for both panels
        box_kws = dict(
            data=df, x='Split type', y=metric, hue='Model type',
            palette=colors, showfliers=False, gap=0.1
        )

        # draw the same plot twice (bottom zoom + top zoom)
        # legend only on the top row, last metric (bottom axis) to place later
        want_legend = i == 0 and m == 0
        sns.boxplot(ax=ax_bot, order=['Random', 'Easy', 'Hard'], legend=want_legend, **box_kws)
        sns.boxplot(ax=ax_top, order=['Random', 'Easy', 'Hard'], legend=False, **box_kws)

        # set the two y-windows
        ax_bot.set_ylim(*lower_ylim)
        ax_top.set_ylim(*upper_ylim)

        # hide touching spines & x labels on top
        ax_top.spines['bottom'].set_visible(False)
        ax_bot.spines['top'].set_visible(False)
        ax_top.set_xticks([])
        ax_top.set_xticks([], minor=True)
        ax_top.tick_params(labelbottom=False)
        ax_top.set_xlabel('')

        # add diagonal “break” ticks
        add_break_marks(ax_top, ax_bot)

        metric_nice = metric_nice_names[metric]

        # titles / labels
        if i == 0:
            ax_top.set_title(metric_nice)
            ax_bot.set_xlabel('')
        else:
            ax_bot.set_xlabel('Data split type')

        ax_bot.set_ylabel(f'{subset}\n{metric_nice}', y=0.65)

        ax_top.set_ylabel('')

        # reference line (draw on both; it will only show if within the panel range)
        # for a in (ax_bot, ax_top):
        #     a.axhline(1, color='k', ls='--', lw=0.8, zorder=0.001)

        # optional: your despine helper on both axes
        despine(ax_bot)
        despine(ax_top)

        if want_legend:
            legend_ax = ax_bot

# # place the legend outside using the saved axis
if legend_ax is not None:
    sns.move_legend(legend_ax, loc='lower center',
                    bbox_to_anchor=(0.5, 0.5), frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
metrics = ['relbias', 'flv', 'fhv']

# --- choose per-metric ranges (tune these as you like) ---
# lower windows focus on the bulk; upper windows show the extreme tail
breaks = {
    m: (
        (-105,  224),                                      # lower y-limits
        (309, 350)       # upper y-limits
    )
    for m in metrics
}

fig, axes = plt.subplots(2, len(metrics), figsize=(10, 6),
                         sharex=False, sharey=False,
                         gridspec_kw={'wspace': 0.4})

legend_ax = None  # we’ll save the bottom axis of the top-right panel for legend

for i, subset in enumerate(['Train', 'Test']):
    for m, metric in enumerate(metrics):

        df = sel_subset(ds.where(ds.has_obs.load(), drop=True), 1 if subset == 'Train' else 3)[metrics].to_dataframe().reset_index()
        parts = df['model'].str.rsplit('_', n=1, expand=True)
        df['Model type'] = parts[0].str.capitalize()
        df['Split type'] = parts[1].str.capitalize()

        # Get breaks.
        lower_ylim, upper_ylim = breaks[metric]

        # replace this slot with broken axes
        bot_h = lower_ylim[1] - lower_ylim[0]
        top_h = upper_ylim[1] - upper_ylim[0]
        ratio = top_h / bot_h
        ax = axes[i, m]
        ax_top, ax_bot = make_broken_axes(fig, ax, hspace=0.04, height_ratios=(ratio, 1))

        colors = {
            'Lumped': "#4F9CFF",
            'Routed': "#DAA507",
        }

        # common kwargs for both panels
        box_kws = dict(
            data=df, x='Split type', y=metric, hue='Model type',
            palette=colors, showfliers=False, showcaps=True, gap=0.1
        )

        # draw the same plot twice (bottom zoom + top zoom)
        # legend only on the top row, last metric (bottom axis) to place later
        want_legend = i == 0 and m == -0
        sns.boxplot(ax=ax_bot, order=['Random', 'Easy', 'Hard'], legend=want_legend, **box_kws)
        sns.boxplot(ax=ax_top, order=['Random', 'Easy', 'Hard'], legend=False, **box_kws)

        # set the two y-windows
        ax_bot.set_ylim(*lower_ylim)
        ax_top.set_ylim(*upper_ylim)

        # hide touching spines & x labels on top
        ax_top.spines['bottom'].set_visible(False)
        ax_bot.spines['top'].set_visible(False)
        ax_top.set_xticks([])
        ax_top.set_xticks([], minor=True)
        # ax_top.tick_params(labelbottom=False)
        ax_top.set_xlabel('')

        # add diagonal “break” ticks
        add_break_marks(ax_top, ax_bot)

        metric_nice = metric_nice_names[metric]

        # titles / labels
        if i == 0:
            ax_top.set_title(metric_nice)
            ax_bot.set_xlabel('')
        else:
            ax_bot.set_xlabel('Data split type')

        ax_bot.set_ylabel(f'{subset}\n{metric_nice}', y=0.6)

        ax_top.set_ylabel('')

        # reference line (draw on both; it will only show if within the panel range)
        for a in (ax_bot, ax_top):
            a.axhline(0, color='k', ls='--', dashes=(5, 5), lw=0.4, zorder=999)

        # optional: your despine helper on both axes
        despine(ax_bot)
        despine(ax_top)

        if want_legend:
            legend_ax = ax_bot

# # place the legend outside using the saved axis
if legend_ax is not None:
    sns.move_legend(legend_ax, loc='lower center',
                    bbox_to_anchor=(0.5, 0.6), frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
metrics_min = ['kge']
metrics_max = ['bias2', 'varerr', 'phaseerr', 'relbias', 'flv', 'fhv']
obs_hrus = ds.where(ds.has_obs.load(), drop=True).hru.values

hrus_met = enrich(
    ds=ds.sel(hru=obs_hrus).isel(time=0).sel(model='routed_random'),
    hrus=hrus.loc[obs_hrus],
    variables=metrics_min + metrics_max + ['subset'] + [m + '_impr' for m in metrics_min + metrics_max]
)

hru_lakes = hrus.loc[ds.where(ds.is_lake.load(), drop=True).hru.values]

hrus = enrich(ds=ds.is_lake.where(ds.is_lake), hrus=hrus, variables='is_lake')

In [ ]:
metrics = ['kge', 'flv', 'fhv']

fig, axes = plt.subplots(2, len(metrics), figsize=(16, 8), subplot_kw={'projection': lv95},
                         gridspec_kw={'hspace': 0.02, 'wspace': 0.02})

bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
    name='admin_0_boundary_lines_land', scale='10m', facecolor='none')

hru_train = hrus_met[hrus_met['subset'].isin([1, 2])]
hru_test = hrus_met[hrus_met['subset'].isin([3])]

for ax in axes.flatten():
    ax.set_global()
    bg.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.8, zorder=0)
    ax.add_feature(bodr, linestyle='--', edgecolor='k', alpha=1, zorder=1, lw=0.7)

for m, metric in enumerate(metrics):

    for i, mode in enumerate(['abs', 'rel']):
        ax = axes[i, m]

        if mode == 'abs':
            if metric == 'kge':
                cmap = sns.color_palette("plasma", as_cmap=True)
                vmin, vmax = 0, 1.0
                extent = 'min'
                unit = '(-)'
            else:
                vmin, vmax = -70, 70
                cmap = sns.color_palette('PiYG', as_cmap=True)
                extent = 'both'
                unit = '(%)'
        else:
            vmin, vmax = -80, 80
            cmap = sns.color_palette("vlag", as_cmap=True)
            extent = 'both'
            unit = '(%)'

        metric_mode = metric if mode == 'abs' else metric + '_impr'
        metric_label = f'{metric.upper()} {"" if mode == "abs" else "Impr. "}{unit}'

        for s, eval_set in enumerate([hru_train, hru_test]):
            marker = 'o' if s == 0 else 'D'
            lw = 0.7 if s == 0 else 1.2
            im = ax.scatter(
                eval_set.centroid.x, eval_set.centroid.y, c=eval_set[metric_mode],
                s=20, vmin=vmin, vmax=vmax, cmap=cmap, edgecolor='k', lw=lw, marker=marker, zorder=2)

            ax.set_title('')

            axins = ax.inset_axes((0.05, 0.95, 0.25, 0.02))
            fig.colorbar(im, cax=axins, orientation='horizontal', label=metric_label, extend=extent)


# Overfitting analysis

To assess overfitting of the 'random' setup, we perform a test: In the validation and test (=evaluation) catchments
of the 'easy' and 'hard'  (='other') setups, we compute, respectively:

* Baseline: Percentage KGE change of the evaluation stations of 'random' versus 'other' setup: This indicates the
performance difference between the setups on the evaluation set.
* Overfitting: Percentage KGE change of the training stations of 'random' versus evaluation set of the 'other' setup:
This indicates how much better the 'random' setup performs on its training set compared to the evaluation set of
the 'other' setup.

We test the abseline and overfitting values using a Mann-Whitney U test. A significant difference would indicate
that the 'random' setup is overfitting to its training set. Further, we compute the difference of the median values,
indicating whether the overfitting effect is larger than the baseline performance difference.

In [ ]:
ds_r = ds.sel(model='routed_random')
ds_e = ds.sel(model='routed_easy')
ds_h = ds.sel(model='routed_hard')

In [ ]:
def overfit_test(a: xr.Dataset, b: xr.Dataset, c: xr.Dataset) -> None:
    """
    Perform a Mann-Whitney U test to assess whether model 'a' overfits. Models 'b' and 'c' are spatial-split trained models, with 'b' being the easier split and 'c' the harder split.

    Args:
        a (xr.Dataset): Dataset of model 'a' (random-split trained model).
        b (xr.Dataset): Dataset of model 'b' (spatial-split easy trained model).
        c (xr.Dataset): Dataset of model 'c' (spatial-split hard trained model).

    """

    fig, axes = plt.subplots(1, 2, figsize=(6, 1.5), sharey=True, gridspec_kw={'wspace': 0.05})

    for i, (o, name) in enumerate(zip([b, c], ['CS-E', 'CS-H'])):

        ax = axes[i]

        # Find HRUs which are in the training set of both models, and those which are in the evaluation set of both models.
        both_eval = ((a.subset.isin([2, 3])) & (o.subset.isin([2, 3]))).compute()

        # Relative gain model a to b for the training and evaluation sets.
        a_kge = a.kge.where(both_eval, drop=True)
        o_kge = o.kge.where(both_eval, drop=True)
        diff = a_kge - o_kge

        wc = wilcoxon(diff, method='asymptotic')

        v = wc.statistic
        p = wc.pvalue
        z = wc.zstatistic
        n = len(diff)
        r = z / np.sqrt(n)

        bins = np.linspace(-.6, .6, 12)
        ax.hist(diff, bins=bins, alpha=0.6, density=False, color='0.5')
        ax.axvline(diff.median().item(), color='tab:red', ls='--', lw=1.4, label='Median', zorder=10)
        ax.axvline(0, color='k', ls='-', lw=1.4, zorder=9)
        ax.set_xlabel('KGE difference', fontsize=8)
        if i == 0:
            ax.set_ylabel('Density', size=8)
        else:
            ax.legend(frameon=False, loc='upper right', fontsize=8)
            ax.set_ylabel('')
        despine(ax)

        ax.text(0.01, 0.95,
                f'Wilcoxon signed\nrank test\n-----------------------\nn={n}\nZ={z:.2f}\np={p:.3f}\nr={r:0.2f}\nMedian={diff.median().item():.2f}',
                ha='left', va='top', transform=ax.transAxes, fontsize=8)

        title = f'Evaluation set KGE difference: RS - {name}'
        
        ax.set_title(title, fontsize=8)

    fig.savefig('./figures/overfit.png', dpi=300, bbox_inches='tight')

overfit_test(ds_r, ds_e, ds_h)

We have shown that the 'random' setup does not overfit on the training set. For the 'hard' evaluation catchments,
the 'hard' setup performs even better than the 'random' setup on its training catchments. However, differences in
median are small (<0.01) in both cases. We can conclude that the overfitting effect is negligible. Thertefore, we
can perform our qualitative analysis on the 'random' setup with including all catchments, i.e., training, validation,
and test, to obtain a complete picture across hydrological Switzerland.

# Climatology

In [ ]:
variables = {
    'e_pred_mmd': ('Evaporation (mm/d)', 'Reds'),
    'q_pred_mmd': ('Runoff (mm/d)', 'Greens'),
    'Q_pred_mmd': ('Discharge (mm/d)', 'Blues'),
}

ds_q = ds[variables.keys()].sel(model='routed_random').load().quantile([0.05, 0.5, 0.95], dim='time').compute()

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, len(variables),
    figsize=(12, 9),
    subplot_kw={'projection': lv95},
    gridspec_kw={'wspace': 0.02, 'hspace': 0.02})

bodr = cartopy.feature.NaturalEarthFeature(
    category='cultural', 
    name='admin_0_boundary_lines_land',
    scale='10m',
    facecolor='none'
)

for i, (key, (name_nice, cmap)) in enumerate(variables.items()):
    for s, scale_nice in enumerate(['5-Percentile', 'Median', '95-Percentile']):

        ds_q_sel = ds_q[key].isel(quantile=s)

        # define log scale bounds
        vmin = ds_q_sel.quantile(0.05).item()
        if vmin == 0:
            vmin = 1e-5
        vmax = ds_q_sel.quantile(0.95).item()

        norm = mpl.colors.LogNorm(vmin=vmin, vmax=vmax, clip=True)

        ax = axes[s, i]
        ax.set_global()
        bg.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.8, zorder=0)
        ax.add_feature(bodr, linestyle='--', edgecolor='k', alpha=1, zorder=1, lw=0.7)

        # plot WITHOUT GeoPandas auto-legend
        enrich(ds=ds_q_sel, hrus=hrus, variables='R_pred').plot(
            column='R_pred',
            cmap=cmap,
            legend=False,     # <-- IMPORTANT
            ec='k',
            lw=0.1,
            alpha=0.6,
            ax=ax,
            norm=norm,
        )

        axins = ax.inset_axes((0.045, 0.95, 0.3, 0.02))
        mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        cbar = plt.colorbar(
            mappable,
            cax=axins,
            orientation='horizontal',   # or 'vertical'
            fraction=0.046,
            pad=0.04,
            extend='both',
            label=f'{name_nice}\n{scale_nice}'
        )

        cbar.ax.xaxis.set_minor_locator(mpl.ticker.LogLocator(subs=np.arange(2, 9) * 0.1))
        cbar.ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
        cbar.ax.tick_params(which='major', labelsize=5)

        # colorbar label size
        cbar.ax.xaxis.label.set_size(7)

        ax.set_title('')


# Trend analysis

In [ ]:
ds_obs = ds.Q_obs_mmd.where(ds.has_obs.load(), drop=True)
all_obs = ds_obs.where(ds_obs.notnull().all('time').load(), drop=True).hru.values

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

for i, ax in enumerate(axes):

    if i == 0:
        ax.set_title('Routed')
        var = 'Q_pred_mmd'
    else:
        ax.set_title('PREVAH')
        var = 'q_prevah_mmd'

    ax.hexbin(
        ds_Q_trend.Q_obs_mmd.sel(hru=all_obs),
        ds_Q_trend[var].sel(hru=all_obs),
        bins='log',
        gridsize=50,
        lw=0.2,
        ec='w',
        extent=[-25, 25, -25, 25],
    )

    # ax.set_xlim(-25, 25)
    # ax.set_ylim(-25, 25)
    ax.set_xlabel('Observed trend (mm/year²)')
    ax.set_ylabel('Predicted trend (mm/year²)')

    ax.plot([-25, 25], [-25, 25], color='k', ls='--', lw=0.8, zorder=10)
    despine(ax)

In [ ]:
def plot_trend_map(trends: xr.Dataset, vars: list[str], titles: list[str], units: str, cmap='BrBG') -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), subplot_kw={'projection': lv95}, gridspec_kw={'wspace': 0.02})

    mx = np.max([np.abs(trends[vars[0]]).quantile(0.8).item(), np.abs(trends[vars[1]]).quantile(0.8).item()])
    mn = -mx
    mx = mx

    for t, var in enumerate(['Q_pred_mmd', 'q_prevah_mmd']):
        ax = axes[t]

        bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
            name='admin_0_boundary_lines_land', scale='10m', facecolor='none')

        ax.set_global()

        bg.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.8, zorder=0)
        ax.add_feature(bodr, linestyle='--', edgecolor='k', alpha=1, zorder=10, lw=0.7)

        axins = ax.inset_axes((0.05, 0.95, 0.25, 0.02))
        im = enrich(
            ds=trends[var],
            hrus=hrus,
            variables='trend'
        ).plot(
            column='trend',
            cmap=cmap,
            legend=True,
            ec='k',
            lw=0.1,
            alpha=0.6,
            vmin=mn,
            vmax=mx,
            ax=ax,
            legend_kwds={'cax': axins, 'orientation': 'horizontal', 'label': f'Trend ({units})'}
        )

        ax.set_title(titles[t])

In [ ]:
ds_Q_annual = ds.sel(model='routed_random')[['Q_pred_mmd', 'q_prevah_mmd', 'Q_obs_mmd']].resample(time='1YE').sum(skipna=True).compute()
ds_Q_annual = ds_Q_annual.assign_coords(time=ds_Q_annual.time.dt.year)
ds_Q_trend = ds_Q_annual.polyfit(dim='time', deg=1, skipna=True).sel(degree=1)
ds_Q_trend = ds_Q_trend.rename({var: var.strip('_polyfit_coefficients') for var in ds_Q_trend.data_vars})
ds_Q_trend = ds_Q_trend.where(ds_Q_trend != 0)

plot_trend_map(
    ds_Q_trend,
    ['Q_pred_mmd', 'q_prevah_mmd'],
    ['Streamflow trend', 'Runoff trend (PREVAH)'],
    units='mm/year/year',
    cmap='BrBG')

In [ ]:
ds_Q_annual_max = ds.sel(model='routed_random')[['Q_pred_mmd', 'q_prevah_mmd', 'Q_obs_mmd']].resample(time='1YE').max(skipna=True).compute()
ds_Q_annual_max = ds_Q_annual_max.assign_coords(time=ds_Q_annual_max.time.dt.year)
ds_Q_trend_max = ds_Q_annual_max.polyfit(dim='time', deg=1, skipna=True).sel(degree=1)
ds_Q_trend_max = ds_Q_trend_max.rename({var: var.strip('_polyfit_coefficients') for var in ds_Q_trend_max.data_vars})

plot_trend_map(
    ds_Q_trend_max,
    ['Q_pred_mmd', 'q_prevah_mmd'],
    ['Streamflow maximum trend', 'Runoff maximum trend (PREVAH)'],
    units='mm/d/year',
    cmap='BrBG'
)

In [ ]:
ds_low_flow_threshold = ds[['Q_pred_mmd', 'q_prevah_mmd', 'Q_obs_mmd']].sel(model='routed_random').sel(time=slice(None, '2002')).load().quantile(0.05, dim='time').compute()
ds_is_low_flow = (ds[['Q_pred_mmd', 'q_prevah_mmd', 'Q_obs_mmd']].sel(model='routed_random') <= ds_low_flow_threshold).compute()
ds_low_flow_count = ds_is_low_flow.resample(time='1YE').sum(skipna=True).compute()
ds_low_flow_count = ds_low_flow_count.assign_coords(time=ds_low_flow_count.time.dt.year)

ds_Q_trend_lf = ds_low_flow_count.polyfit(dim='time', deg=1, skipna=True).sel(degree=1)
ds_Q_trend_lf = ds_Q_trend_lf.rename({var: var.strip('_polyfit_coefficients') for var in ds_Q_trend_lf.data_vars})

ds_Q_trend_lf = ds_Q_trend_lf.where(ds_Q_trend_lf != 0)

plot_trend_map(
    ds_Q_trend_lf,
    ['Q_pred_mmd', 'q_prevah_mmd'],
    ['Lowflow (Q5 w.r.t. 1993-2022) days trend', 'Lowflow (Q5 w.r.t. 1993-2022) days trend (PREVAH)'],
    units='d/year',
    cmap='BrBG_r'
)

## Time series analysis

In [ ]:
num_plot = 30
time_subset = slice('2007', '2009')
# time_subset = slice(None, None)

# Test set
plot_indices = ds.hru.where(ds.has_obs.load(), drop=True).values
plot_indices = plot_indices[:num_plot]

fig, axes = plt.subplots(len(plot_indices), figsize=(18, len(plot_indices) * 1.5), sharex=True, squeeze=False)

axes = axes[:, 0]

for i, obs_i in tqdm(enumerate(plot_indices), total=len(plot_indices)):
    ax = axes[i]
    ds_subset = ds.sel(hru=obs_i)
    ds_subset = ds.sel(hru=obs_i, time=time_subset)
    ds_subset.Q_obs_m3s.plot(ax=ax, color='0.3', alpha=1, ls='-', lw=1.5, label='Observed')
    ds_subset.sel(model='routed_random').Q_pred_m3s.plot(ax=ax, color='tab:blue', alpha=0.8, lw=1.1, label='Semi-distributed (static)')
    ds_subset.sel(model='lumped_random').Q_pred_m3s.plot(ax=ax, color='tab:orange', alpha=0.8, lw=1.1, label='Lumped')

    ax.set_title('')

    upstream_area = ds_subset.upstream_area.values.item()
    nse_d = ds_subset['nse'].sel(model='routed_random').item()
    nse_l = ds_subset['nse'].sel(model='lumped_random').item()
    ax.text(0.01, 0.99,
        f'HRU {ds_subset.hru.item()}; upstream area = {upstream_area:0.0f} km2',
        transform=ax.transAxes, ha='left', va='top')
    ax.text(0.95, 0.99,
        f'NSE (dirst) = {nse_d:0.2f}\nNSE (lumped) = {nse_l:0.2f}',
        transform=ax.transAxes, ha='right', va='top')

    # ax.text(0.01, 0.89, ds_l_subset.is_lake.item(), transform=ax.transAxes, ha='left', va='top',)

    ax.set_xlabel('Time' if i == (len(plot_indices) - 1) else '')

    if i == 0:
        ax.legend(frameon=False, loc='upper center', ncol=3)
        ax.set_ylabel('Q (m3 s-1)')
    else:
        ax.set_ylabel('')

    despine(ax)

fig.align_ylabels(axes)

# savefig(fig, '../../viz/misc/time_series.pdf', dpi=300)

In [ ]:
ds_sel = ds.where(ds.has_obs.load(), drop=True)

ds_sel = ds_sel.sel(time=slice('2006', '2008'))

ds_sel['x'] = ds.x.sel(hru=ds.hru.where(ds.has_obs, drop=True)).values
ds_sel['y'] = ds.y.sel(hru=ds.hru.where(ds.has_obs, drop=True)).values

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import json
from bokeh import palettes as bpalettes
from bokeh.plotting import figure, show
import xyzservices.providers as xyz
from bokeh.models import ColumnDataSource, CustomJS, ColorBar
from bokeh.layouts import column
from bokeh.io import output_notebook
from pyproj import Transformer
from bokeh.models import LinearColorMapper

PLOT_VARS = {}

# cmap = plt.get_cmap('Paired')
# color_vals = cmap(np.linspace(0, 1, len(ds_sel.model.values)))
# colors = [mpl.colors.to_hex(color) for color in color_vals]
colors = ['#882255', '#CC6677', '#DDCC77', '#117733', '#44AA99', '#88CCEE']

for m, model in enumerate(ds_sel.model.values):
    PLOT_VARS[f'Q_pred_{model}_m3s'] = {
        'var': 'Q_pred_m3s',
        'legend_label': f'Predicted runoff ({model}) (m³/s)',
        'color': colors[m],
        'alpha': 0.8,
        'line_width': 2,
        'model': model
    }

# PLOT_VARS['Q_pred_instant_m3s'] = {
#     'legend_label': 'Predicted runoff, instant (m³/s)',
#     'color': '#7eb963',
#     'alpha': 0.8,
#     'line_width': 1.5,
# }

PLOT_VARS['Q_obs_m3s'] = {
    'var': 'Q_obs_m3s',
    'legend_label': 'Observed runoff  (m³/s)',
    'color': '#000000',
    'alpha': 0.8,
    'line_width': 1.5,
    'model': None
}

transformer = Transformer.from_crs("EPSG:2056", "EPSG:3857")

x, y = transformer.transform(ds_sel['x'].values, ds_sel['y'].values)

output_notebook()

cmap = bpalettes.diverging_palette(bpalettes.Reds256, bpalettes.Blues256, n=20)
color_mapper = LinearColorMapper(palette=cmap, low=0, high=1)

# === Prepare Bokeh map source ===
source_data = {
    'id': list(range(len(ds_sel['hru'].values))),
    'x': x.tolist(),
    'y': y.tolist(),
    'label': ds_sel['hru'].values.tolist(),
    'color': [color_mapper.palette[int(val * (len(color_mapper.palette) - 1))] for val in ds_sel.sel(model='routed_random').nnse.values]
}
source = ColumnDataSource(data=source_data)
ts_source = ColumnDataSource(
    data={
        **{'t': []},
        **{key: [] for key in PLOT_VARS.keys()}
        }
    )

# === Convert xarray time series to JS-friendly format ===
ts_data = {}
for i, hru in enumerate(ds_sel['hru'].values):
    ts = ds_sel.sel(hru=hru)
    times = pd.to_datetime(ts['time'].values).astype('int64') // 10**6  # ms since epoch
    d = {'t': times.tolist()}
    for var, info in PLOT_VARS.items():
        var_sel = ts[info['var']]
        if info['model'] is not None:
            var_sel = var_sel.sel(model=info['model'])
        d[var] = var_sel.values.tolist()
    ts_data[int(i)] = d

# === Serialize to JSON ===
ts_data_json = json.dumps(ts_data)

# === Create map plot ===
map_plot = figure(x_axis_type="mercator", y_axis_type="mercator",
                  width=800, height=400, title="Click a Point")
# map_plot.add_tile(Vendors.OSM)
# map_plot.add_tools(TapTool())
map_plot.add_tile(xyz.OpenStreetMap.Mapnik)
scatter = map_plot.scatter(
    x='x',
    y='y',
    size=8,
    source=source,
    fill_color="color",
    line_color='black',
    alpha=0.8)

selected_circle = scatter.glyph.clone()
selected_circle.line_width = 3
nonselected_circle = scatter.glyph.clone()
scatter.selection_glyph = selected_circle
scatter.nonselection_glyph = nonselected_circle

color_bar = ColorBar(
    color_mapper=color_mapper,
    label_standoff=12,
    border_line_color=None,
    location=(0,0),
    title="Normalized NSE (-)",    # <- your title text
    title_text_align="center", # left / center / right
    title_text_font_style="italic",
    title_text_font_size="11pt",
)
map_plot.add_layout(color_bar, 'right')

# === Create time series plot ===
ts_plot = figure(width=800, height=450, x_axis_type='datetime', title='Time Series')

for var, info in PLOT_VARS.items():
    ts_plot.line(
        't', var, source=ts_source, **{key: value for key, value in info.items() if key not in ['var', 'model']})

ts_plot.legend.click_policy = "hide"

# Div to show selected point info
# div = Div(width=400)

# === JS callback: uses selected.indices like your working version ===
callback = CustomJS(args=dict(source=source, ts_source=ts_source, ts_plot=ts_plot), code=f"""
    const ts_all = {ts_data_json};

    const click_x = cb_obj.x;
    const click_y = cb_obj.y;

    const xs = source.data['x'];
    const ys = source.data['y'];
    const ids = source.data['id'];
    const labels = source.data['label'];

    let closest_idx = -1;
    let min_dist = Infinity;

    for (let i = 0; i < xs.length; i++) {{
        const dx = xs[i] - click_x;
        const dy = ys[i] - click_y;
        const dist = dx * dx + dy * dy;
        if (dist < min_dist) {{
            min_dist = dist;
            closest_idx = i;
        }}
    }}

    if (closest_idx === -1) return;

    source.selected.indices = [closest_idx];

    const id = ids[closest_idx];
    const label = labels[closest_idx];
    const ts = ts_all[id];
    if (!ts) return;

    ts_source.data = {{ t: ts.t, {', '.join([f'{var}: ts.{var}' for var in PLOT_VARS.keys()])} }};
    ts_source.change.emit();

    ts_plot.title.text = `Time Series for HRU ${{label}}`;

""")

map_plot.js_on_event('tap', callback)

# === Show everything ===
show(column(map_plot, ts_plot))


In [ ]:
# 121002

dss = ds.sel(hru=175387)

In [ ]:
dss.S_pred_m3.sel(model='routed_random').plot()

In [ ]:
dss.R_pred.sel(model='routed_random').plot()

In [ ]:
dss.Q_pred_m3s.sel(model='routed_random').plot()
dss.Q_pred_m3s.sel(model='routed_easy').plot()

In [ ]:
import torch

In [ ]:
study = optuna.load_study(
        study_name=None,
        storage=f'sqlite:////net/argon/landclim/kraftb/machflow_routing/logs/routed/random/tune/optuna.db')
study.best_trial

In [ ]:
study = optuna.load_study(
        study_name=None,
        storage=f'sqlite:////net/argon/landclim/kraftb/machflow_routing/logs/routed/easy/tune/optuna.db')
study.best_trial

In [ ]:
study = optuna.load_study(
        study_name=None,
        storage=f'sqlite:////net/argon/landclim/kraftb/machflow_routing/logs/routed/hard/tune/optuna.db')
study.best_trial

In [ ]:
rs = torch.load('/net/argon/landclim/kraftb/machflow_routing/logs/routed/random/tune/trial_018/checkpoints/best.ckpt')['state_dict']
cse = torch.load('/net/argon/landclim/kraftb/machflow_routing/logs/routed/easy/tune/trial_001/checkpoints/best.ckpt')['state_dict']
csh = torch.load('/net/argon/landclim/kraftb/machflow_routing/logs/routed/hard/tune/trial_001/checkpoints/best.ckpt')['state_dict']

In [ ]:
for k, _ in rs.items():
    print(k)
    print(rs[k])
    print(cse[k])
    print(csh[k])
    print('-' * 20)

In [ ]:
time = slice('2022', '2025')
time = slice(None, None)

# rhein = 118438
aare = 145830

ds_s = ds.sel(hru=aare, time=time)

ds_s.Q_obs_m3s.plot(color='k', lw=1.5)
ds_s.sel(model='routed_random').Q_pred_m3s.plot(lw=0.8, alpha=0.8)
ds_s.sel(model='lumped_random').Q_pred_m3s.plot(lw=0.8, alpha=0.8)

In [ ]:
hru_i = np.argwhere(ds.has_obs.values).flatten()

In [ ]:
dss = ds.isel(hru=hru_i).sel(model='routed_random')

In [ ]:
q_low = 0.0
q_up = 0.05

dim = 'time'

In [ ]:
obs = dss.Q_obs_m3s.squeeze()
mod = dss.Q_pred_m3s.squeeze()
mask = (obs.notnull() & mod.notnull()).compute()
obs_da = obs.where(mask, drop=True).compute()
mod_da = mod.where(mask, drop=True).compute()

In [ ]:
drop_empty = True
mod = ds.Q_pred_m3s
obs = ds.Q_obs_m3s

only_mod_dims = tuple(np.setdiff1d(np.array(mod.dims), np.array(obs.dims)))
mask = mod.notnull().any(only_mod_dims) & obs.notnull()

if drop_empty:
    all_miss = mask.any(dim).compute()
    mask = mask.where(all_miss, drop=True)

obs_da = obs.where(mask).compute()
mod_da = mod.where(mask).compute()

In [ ]:
tuple(np.setdiff1d(np.array(mod.dims), np.array(obs.dims)))

In [ ]:
only_mod_dims

In [ ]:
(1, 2, 3,)[1:]

In [ ]:
only_mod_dims = tuple(np.setdiff1d(np.array(mod.dims), np.array(obs.dims)))
n_only_mod_dims = len(only_mod_dims)

obs_da = obs_da.transpose(..., dim)
mod_da = mod_da.transpose(*only_mod_dims, ..., dim)

da_res = mod_da.isel({dim: 0}).copy()

for idx in np.ndindex(mod_da.shape[:-1]):

    mod_i = mod_da[idx].values

    obs_i = obs_da[idx[n_only_mod_dims:]].values

    sort_idx = np.argsort(obs_i)

    n = np.isfinite(obs_i).sum()
    low_i = int(np.floor(q_low * n))
    up_i = int(np.floor(q_up * n))
    sort_idx_in_range = sort_idx[low_i:up_i]

    obs_in_range = obs_i[sort_idx_in_range]
    mod_in_range = mod_i[sort_idx_in_range]

    perc_bias = 100 * (mod_in_range - obs_in_range).sum() / obs_in_range.sum()

    da_res[idx] = perc_bias

da_res.attrs['units'] = '%'

In [ ]:
q_low = 0.0
q_up = 0.05

dim = 'time'

obs_da = obs_da.transpose(..., dim)
mod_da = mod_da.transpose(..., dim)

result_da = obs_da.isel({dim: 0}).copy()

for idx in np.ndindex(obs_da.shape[:-1]):

    obs_i = obs_da[idx].values
    mod_i = mod_da[idx].values

    sort_idx = np.argsort(obs_i)

    n = np.isfinite(obs_i).sum()
    low_i = int(np.floor(q_low * n))
    up_i = int(np.floor(q_up * n))
    sort_idx_in_range = sort_idx[low_i:up_i]

    obs_in_range = obs_i[sort_idx_in_range]
    mod_in_range = mod_i[sort_idx_in_range]

    perc_bias = 100 * (mod_in_range - obs_in_range).sum() / obs_in_range.sum()

    result_da[idx] = perc_bias

# n = len(sort_idx)
# low_i = int(np.floor(q_low * n))
# up_i = int(np.floor(q_up * n))
# sort_idx_in_range = sort_idx[low_i:up_i]

# obs_in_range = obs.values[sort_idx_in_range]
# mod_in_range = mod.values[sort_idx_in_range]

# perc_bias = 100 * (mod_in_range - obs_in_range).sum() / obs_in_range.sum()

In [ ]:
result_da.plot.hist(bins=np.linspace(-100, 1000, 101))

In [ ]:
from glob import glob

In [ ]:
# Load predictions.
ds = xr.open_zarr('/net/argon/landclim/kraftb/machflow_routing/data/results/predictions.zarr').sel(
    time=slice('1993', None))
ds = ds.where(ds.has_obs.load(), drop=True)

metrics_max = ['kge', 'nse', 'r']


# Compute metrics and add them to dataset.
met = compute_metrics(ds.Q_obs_m3s, ds.Q_pred_m3s, metrics=metrics_min + metrics_max, dim='time')
ds = xr.merge([ds, met])
ds_geom = gpd.GeoDataFrame(geometry=gpd.points_from_xy(ds.x, ds.y)).set_crs(epsg=2056)

In [ ]:
ids = [f[-8:-4] for f in glob('/net/argon/landclim/kraftb/machflow/data/camels_ch/timeseries/simulation_based/*')]

In [ ]:
ids_subset = []
nses = []

for i, id in enumerate(ids):
    d_sim = pd.read_csv(
        f'/net/argon/landclim/kraftb/machflow/data/camels_ch/timeseries/simulation_based/CAMELS_CH_sim_based_{id}.csv')['discharge_vol_sim(m3/s)'].values
    d_obs = pd.read_csv(f'/net/argon/landclim/kraftb/machflow/data/camels_ch/timeseries/observation_based/CAMELS_CH_obs_based_{id}.csv')['discharge_vol(m3/s)'].values
    mask = np.isfinite(d_obs)
    if not np.any(mask):
        continue
    d_sim = d_sim[mask]
    d_obs = d_obs[mask]
    nse = 1 - np.sum((d_sim - d_obs) ** 2) / np.sum((d_obs - np.mean(d_obs)) ** 2)
    ids_subset.append(id)
    nses.append(nse)

In [ ]:
dsc

In [ ]:
camels_nse = pd.DataFrame({'id': ids_subset, 'nse_sim': nses}).set_index('id')

In [ ]:
camels_nse.median()

In [ ]:
camels = gpd.read_file('/net/argon/landclim/kraftb/machflow/data/camels_ch/catchment_delineations/CAMELS_CH_gauging_stations.shp')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

camels.plot(ax=ax)
ds_geom.plot(ax=ax, color='red', markersize=1)

In [ ]:
camels_nse['is_ind_ds'] = False

for i in range(len(camels)):
    cs = camels.iloc[i]
    is_close = ds_geom.distance(cs.geometry) < 1000
    gid = int(cs.gauge_id)
    if is_close.sum() == 0:
        continue
    elif is_close.sum() > 1:
        print('Multiple close points found!')
    else:
        print('One close point found!')
        camels_nse.loc[str(gid), 'is_ind_ds'] = True
        

In [ ]:
camels_nse.nse_sim.median()

In [ ]:
camels_nse[camels_nse.is_ind_ds].nse_sim.median()

In [ ]:
camels = pd.read_csv('/net/argon/landclim/kraftb/machflow/data/camels_ch/timeseries/simulation_based/CAMELS_CH_sim_based_2007.csv')[4017:][['discharge_vol_sim(m3/s)']]

In [ ]:
camels = pd.read_csv('/net/argon/landclim/kraftb/machflow/data/camels_ch/timeseries/observation_based/CAMELS_CH_obs_based_2004.csv')[4017:][['date', 'discharge_vol(m3/s)']]

In [ ]:
dsc_list = []

dsc = xr.open_zarr('/net/argon/landclim/kraftb/machflow/runs/staticall_allbasins_sqrttrans/LSTM/xval/fold_000/preds.zarr').sel(
time=slice('1993', None))
# dsc = dsc.where(dsc.cv_set.load() > 0, drop=True)
# dsc_list.append(dsc)
# dsc = xr.concat(dsc_list, dim='station')

met = compute_metrics(dsc.Qm3s, dsc.Qm3s_prevah, metrics=metrics_min + metrics_max, dim='time')
dsc = xr.merge([dsc, met])

In [ ]:
# Load HRU polygons.
hrus = gpd.read_file(
    '/net/argon/landclim/kraftb/machflow_routing/data/processed/hrus/hrus.gpkg', use_arrow=True).set_index('EZGNR')

In [ ]:
# Load HRU polygons.
cm = gpd.read_file(
    '/net/argon/landclim/kraftb/machflow/data/obs/shapefile/stations_tabsd.shp', use_arrow=True).set_index('mach_ID')
cm = cm.to_crs(hrus.crs)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
camels.plot(ax=ax)
# cm.plot(ax=ax, color='red')
# ds_geom.plot(ax=ax, color='brown')

ax.scatter(ds.x, ds.y, color='red', s=6)

In [ ]:
ds_geom = gpd.GeoDataFrame(geometry=gpd.points_from_xy(ds.x, ds.y)).set_crs(epsg=2056)

In [ ]:
for i in range(len(camels)):
    is_close = ds_geom.distance(camels.geometry.iloc[i]) < 1000
    if is_close.sum() == 0:
        continue
    elif is_close.sum() > 1:
        print('Multiple close points found!')
    else:
        print('One close point found!')

In [ ]:
cm

In [ ]:
cm.owner_ID.isin([int(id) for id in camels.gauge_id])

In [ ]:
cm

In [ ]:
cm.loc[[f'CHFO-{id:0.0f}' for id in camels.gauge_id]]

In [ ]:
cm

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
hrus.plot(ax=ax)
cm.plot(ax=ax, color='red')

ax.scatter(ds.x, ds.y, color='pink', s=6)

In [ ]:
hrus

In [ ]:
cm_joined = gpd.sjoin(hrus, cm, predicate="contains")

In [ ]:
sum([h in cm_joined.mach_ID for h in dsc.station.values])

In [ ]:
cm_joined.mach_ID

In [ ]:
dsc.station.values

In [ ]:
cm_joined

In [ ]:
sum([h in ds.hru for h in cm_joined.index])

In [ ]:
cm_joined.mach_ID.isin(dsc.station.values).sum()

In [ ]:
cm_joined[cm_joined.mach_ID.isin(dsc.station)]

In [ ]:
dsc

In [ ]:
dsc

In [ ]:
hrus.columns

In [ ]:
dsc

In [ ]:
met = compute_metrics(dsc.Qmm, dsc.Qmm_mod, metrics=metrics_min + metrics_max, dim='time')
dsc = xr.merge([dsc, met])

In [ ]:
dsc

In [ ]:
ds

In [ ]:
camels = gpd.read_file('/net/argon/landclim/kraftb/machflow/data/camels_ch/catchment_delineations/CAMELS_CH_gauging_stations.shp', use_arrow=True)
camels['gauge_id'] = camels['gauge_id'].astype(int)

In [ ]:
hydroch = gpd.read_file('/net/argon/landclim/kraftb/machflow_routing/data/processed/hydro_ch/Inventory_discharge_hydroCH_manually_cleaned.gpkg', use_arrow=True)

In [ ]:
camel_ids = []

for id6 in hydroch.ID6:
    idx = np.argwhere(camels.ID6 == id6)
    if len(idx) == 0:
        print('Not found: ', id6)
        continue

    idx = idx.item()
    camel_ids.append(int(camels.gauge_id[idx]))


In [ ]:
camels_ids = np.asarray(['2009', '2011', '2016', '2018', '2019', '2020', '2024', '2029', '2030', '2033', '2044', '2053', '2056', '2063', '2067', '2078', '2084', '2085', '2086', '2087', '2091', '2099', '2102', '2104', '2106', '2110', '2117', '2119', '2135', '2143', '2152', '2160', '2161', '2174', '2176', '2181', '2205', '2210', '2215', '2239', '2243', '2247', '2265', '2270', '2288', '2289', '2342', '2346', '2351', '2352', '2364', '2368', '2370', '2372', '2378', '2387', '2418', '2419', '2420', '2446', '2457', '2468', '2473', '2475', '2481', '2494', '2498', '2602', '2606', '2620', '3001', '3006', '3033', '4010', '5030', '5032', '6005', '6007', '6011']).astype(int)

In [ ]:
camels_ids

In [ ]:
camels.set_index('gauge_id').loc[camels_ids]

In [ ]:
eval_ids = []

for id6 in camels.set_index('gauge_id').loc[camels_ids].ID6:
    idx = np.argwhere(hydroch.ID6 == id6)
    if len(idx) == 0:
        print('Not found: ', id6)
        continue

    eval_ids.extend(idx.flatten())


In [ ]:
hydroch_subset = hydroch.iloc[eval_ids]
hydroch_subset

In [ ]:
cm_overlay = gpd.sjoin(hrus, hydroch_subset, how='inner', predicate='contains')

In [ ]:
fig, ax = plt.subplots()

cm_overlay.plot(ax=ax)
ds_obs = ds.where(ds.has_obs.load(), drop=True)
ax.scatter(ds_obs.x, ds_obs.y, s=5, color='red')

In [ ]:
ds_subset = ds.sel(hru=cm_overlay.reset_index().EZGNR.values)

In [ ]:
ds_subset.kge.median('hru')